# MFI: Modern Fortran Interfaces — GPU/CuBLAS Test

**Repo:** https://github.com/14NGiestas/mfi/tree/qwen3-cublas

Sets up the Fortran + CUDA toolchain on Colab, clones `qwen3-cublas`, builds with cuBLAS, and runs tests.

**Runtime:** Select **GPU** (T4) before running.

In [ ]:
# 1. Verify GPU is available
!nvidia-smi

In [ ]:
# 2. Install Fortran toolchain + BLAS/LAPACK
!apt-get update -qq && apt-get install -y -qq \
    gfortran libblas-dev liblapack-dev libopenblas-dev \
    python3-pip 2>&1 | tail -3
!pip install -q fypp

# Install fpm (Fortran Package Manager) from binary release
!curl -sL https://github.com/fortran-lang/fpm/releases/download/v0.10.1/fpm-0.10.1-linux-x86_64 \
    -o /usr/local/bin/fpm && chmod +x /usr/local/bin/fpm

!echo '--- versions ---'
!gfortran --version | head -1
!fpm --version
!fypp --version

In [ ]:
# 3. Diagnose CUDA / cuBLAS availability
import os, subprocess, glob

print("=== CUDA DRIVER ===")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
for line in result.stdout.split('\n')[:5]:
    print(line)

print("\n=== CUDA TOOLKIT ===")
cuda_candidates = glob.glob("/usr/local/cuda*") + glob.glob("/usr/lib/cuda*")
for p in cuda_candidates:
    print(f"  Found: {p}")

print("\n=== cuBLAS libraries ===")
!ldconfig -p 2>/dev/null | grep cublas || echo "  (not in ldconfig)"
!find /usr -name "libcublas*" 2>/dev/null | head -5

print("\n=== nvcc ===")
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "  nvcc NOT FOUND")

In [ ]:
# 4. If cuBLAS dev is missing, install CUDA toolkit
import subprocess
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
has_nvcc = r.returncode == 0

if not has_nvcc:
    print("Installing CUDA toolkit...")
    !apt-get install -y -qq cuda-toolkit 2>&1 | tail -3
    r2 = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    if r2.returncode != 0:
        print("cuda-toolkit not in apt, trying libcublas-dev...")
        !apt-get install -y -qq libcublas-dev 2>&1 | tail -3
    print("\n=== After install ===")
    !nvcc --version 2>&1 | head -4
    !ldconfig -p 2>/dev/null | grep cublas | head -3
else:
    print("✅ CUDA toolkit already installed")
    !nvcc --version | head -4

In [ ]:
# 5. Clone MFi qwen3-cublas branch
%cd /content
!rm -rf mfi
!git clone -b qwen3-cublas --single-branch --depth 1 \
    https://github.com/14NGiestas/mfi.git
%cd mfi
!echo '--- repo contents ---'
!ls -la
!echo '--- fpm.toml ---'
!cat fpm.toml

In [ ]:
# 6. Preprocess .fpp -> .f90 with cuBLAS enabled
%cd /content/mfi
!make clean 2>/dev/null
!make FYPPFLAGS="-DMFI_EXTENSIONS -DMFI_USE_CUBLAS"
print("\n✅ Preprocessing complete")

In [ ]:
# 7. Build with fpm
%cd /content/mfi
import os

# Find cuBLAS paths dynamically
cublas_lib = None
for p in ["/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/lib64", "/usr/lib/cuda/lib64"]:
    if os.path.exists(f"{p}/libcublas.so"):
        cublas_lib = p
        break

cuda_include = None
for p in ["/usr/local/cuda/include", "/usr/include"]:
    if os.path.exists(f"{p}/cublas_v2.h"):
        cuda_include = p
        break

print(f"cuBLAS lib path: {cublas_lib}")
print(f"CUDA include path: {cuda_include}")

flags = ""
if cuda_include:
    flags += f"-I{cuda_include} "

if cublas_lib:
    os.environ["LD_LIBRARY_PATH"] = f"{cublas_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"

print(f"\nBuilding with fpm (flags: {flags.strip() or 'none'})...")
!fpm build --flag "{flags.strip()}" 2>&1

In [ ]:
# 8. GPU verification test — fails if GPU is NOT used
%%writefile /content/mfi/test/gpu_verify.f90
program gpu_verify
    use iso_fortran_env
    use iso_c_binding
    use mfi_blas, only: mfi_execution_init, mfi_sgemv, mfi_sgemm, &
                        mfi_get_execution_mode, MFI_USE_CUBLAS
    implicit none
    real(real32), target :: M(64,64), X(64), Y(64), Y_ref(64)
    real(real32), target :: A(32,32), B(32,32), C(32,32), C_ref(32,32)
    real(real32) :: alpha, beta
    integer :: i, j, k
    logical :: all_pass

    call mfi_execution_init()
    print *, "Execution mode:", trim(mfi_get_execution_mode())
    print *, "MFI_USE_CUBLAS:", MFI_USE_CUBLAS

    if (MFI_USE_CUBLAS /= 1) then
        print *, "ERROR: MFI_USE_CUBLAS not enabled at runtime"
        error stop 1
    end if

    all_pass = .true.

    ! === GEMV GPU test ===
    call random_number(M)
    call random_number(X)
    Y = 0.0
    Y_ref = 0.0
    alpha = 2.0
    beta = 0.5

    do i = 1, 64
        do j = 1, 64
            Y_ref(i) = Y_ref(i) + alpha * M(i,j) * X(j)
        end do
        Y_ref(i) = Y_ref(i) + beta * Y(i)
    end do

    Y = 0.0
    call mfi_sgemv(M, X, Y, alpha=alpha, beta=beta, trans='N')

    if (all(abs(Y - Y_ref) < 1.0e-3_real32)) then
        print *, "PASS: GEMV GPU"
    else
        print *, "FAIL: GEMV GPU — max diff =", maxval(abs(Y - Y_ref))
        all_pass = .false.
    end if

    ! === GEMM GPU test ===
    call random_number(A)
    call random_number(B)
    C = 0.0
    C_ref = 0.0
    alpha = 1.0
    beta = 0.0

    do i = 1, 32
        do j = 1, 32
            do k = 1, 32
                C_ref(i,j) = C_ref(i,j) + A(i,k) * B(k,j)
            end do
        end do
    end do

    C = 0.0
    call mfi_sgemm(A, B, C, alpha=alpha, beta=beta)

    if (all(abs(C - C_ref) < 1.0e-2_real32)) then
        print *, "PASS: GEMM GPU"
    else
        print *, "FAIL: GEMM GPU — max diff =", maxval(abs(C - C_ref))
        all_pass = .false.
    end if

    if (.not. all_pass) then
        print *, "GPU tests FAILED"
        error stop 1
    end if
    print *, "ALL GPU TESTS PASSED"
end program

In [ ]:
# 9. Compile and run GPU verification test
%cd /content/mfi
import os, subprocess
os.environ["MFI_USE_CUBLAS"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:/usr/local/cuda/lib64/stubs:{os.environ.get('LD_LIBRARY_PATH', '')}"

!fpm build --flag "-I/usr/local/cuda/include" 2>&1

result = subprocess.run(
    "gfortran -DMFI_EXTENSIONS -DMFI_USE_CUBLAS "
    "-I/usr/local/cuda/include -I./src/mfi -I./src/f77 -I. "
    "test/gpu_verify.f90 src/f77/blas.f90 src/mfi/blas.f90 "
    "-lblas -llapack -lcublas -lcudart "
    "-L/usr/local/cuda/lib64 -Wl,-rpath,/usr/local/cuda/lib64 "
    "-o gpu_verify 2>&1", shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("COMPILE FAILED:")
    print(result.stderr)
else:
    result = subprocess.run(
        "./gpu_verify", shell=True, capture_output=True, text=True,
        env={**os.environ})
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("GPU verification FAILED")
    else:
        print("GPU verification PASSED")

In [ ]:
# 10. Run full test suite with GPU enabled
%cd /content/mfi
import os
os.environ["MFI_USE_CUBLAS"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:/usr/local/cuda/lib64/stubs:{os.environ.get('LD_LIBRARY_PATH', '')}"

!fpm test 2>&1